# The MPC's model: no slip, with an affine grip limit

This is the model the MPC predicts with. It is a simplified version of the robot in `RobotSimulationOpenLoopModel_Merged.ipynb` (the plant), built from the same `simulation_parameters`, and it is the only place the MPC's model is defined. `RobotSimulationMPC.ipynb` loads it from here by cell id.

* **State** $\mathbf{x} = [X, Y, \theta, v_x, v_y, \dot{\theta}]$: the pose, and the velocity along the robot's own axes. **Input** $\mathbf{u}$: the four motor voltages.
* **What it keeps from the plant:** the wheel layout, mass and inertia, the motor, the wheels' inertia, the load transfer (weight shifting between wheels as the robot accelerates), the rolling drag and the roller friction.
* **What it simplifies:** the wheels roll without slipping (no bristles), the motor current settles at once, and three slowly changing values are held over the 0.4 s horizon.
* **What it gives:** the motion, and for each wheel the friction force it needs from the floor and how hard it presses on the floor. Both are affine in $\mathbf{x}$ and $\mathbf{u}$, so the grip limit $-\mu_{lim} N \le F \le \mu_{lim} N$ is a convex constraint.

In [ ]:
# ===========================================================================
# The plant's parameters and robot class (the model takes its geometry and values from them)
# ===========================================================================
import json
import numpy as np
from scipy.linalg import expm

PLANT_NOTEBOOK = "RobotSimulationOpenLoopModel_Merged.ipynb"
plant_cells = {c.get("id"): "".join(c["source"]) for c in json.load(open(PLANT_NOTEBOOK, encoding="utf-8"))["cells"]}
exec(plant_cells["constants"])     # simulation_parameters, GRAVITY
exec(plant_cells["plant"])         # OmniMotorState, OmniRobotSimulation
print(f"Loaded the parameters and robot class from {PLANT_NOTEBOOK}")

## The MPC's model and the grip limit

<details>
<summary><b> Open this for the equations</b></summary>

**State and input:** $\mathbf{x} = [X, Y, \theta, v_x, v_y, \dot{\theta}]^\top$: the pose, and the velocity **along the robot's own axes** $\mathbf{v} = \mathbf{R}(\theta)^\top \mathbf{\dot{X}}_g$ (the last three entries of $\mathbf{x}$). Input $\mathbf{u}$: the four motor voltages.

**Held over the horizon** (worked out once per MPC step from the estimated state, then constants):
* the rolling drag per unit load, $c_i = C_{rr} \tanh(\omega_i / \varepsilon)$ (Step 5.3),
* the roller force per unit load, $\phi_{a,i} = -\mu_{a,i} \frac{2}{\pi} \arctan(k_a v_{a,i})$ (Step 5.8),
* the turning rate $\bar{\dot{\theta}}$ in the turning of the robot's axes, $\frac{d\mathbf{v}}{dt} = \mathbf{a} + \bar{\dot{\theta}}\, \mathbf{S} \mathbf{v}$, with $\mathbf{S} = \begin{bmatrix} 0 & 1 & 0 \\ -1 & 0 & 0 \\ 0 & 0 & 0 \end{bmatrix}$.

With these held, everything below is affine in $\mathbf{x}$ and $\mathbf{u}$. (Only the velocity part $\mathbf{v}$ of $\mathbf{x}$ enters the forces; the pose enters only the geometry at the end.)

**No slip:** $\omega_i = (\mathbf{H}_{drive} \mathbf{v})_i / r$, and the motor current is settled: $i_i = (u_i - K_b \omega_i) / R$. Both affine.

**Acceleration with the loads of this step** (Section 4: $\mathbf{F}_n = \mathbf{F}_{n,\text{static}} + \mathbf{T} \mathbf{a}$), one linear solve with a constant matrix:
$$\underbrace{\left(\mathbf{M} + \tfrac{J_w}{r^2} \mathbf{H}_{drive}^\top \mathbf{H}_{drive} + \mathbf{H}_{drive}^\top \text{diag}(\mathbf{c}) \mathbf{T} - \mathbf{H}_{axial}^\top \text{diag}(\boldsymbol{\phi}_a) \mathbf{T}\right)}_{\text{constant}} \mathbf{a} = \mathbf{H}_{drive}^\top\left(\tfrac{K_t}{r}\mathbf{i} - \mathbf{c} \circ \mathbf{F}_{n,\text{static}}\right) + \mathbf{H}_{axial}^\top \left(\boldsymbol{\phi}_a \circ \mathbf{F}_{n,\text{static}}\right) - \tfrac{J_w}{r^2} \mathbf{H}_{drive}^\top \mathbf{H}_{drive}\, \bar{\dot{\theta}}\, \mathbf{S} \mathbf{v}$$
The right side is affine in $\mathbf{x}$ and $\mathbf{u}$, so $\mathbf{a}$, $\frac{d\mathbf{v}}{dt}$ and $\mathbf{F}_n$ are affine.

**Floor force each wheel needs** (the wheel equation, Step 5.7, solved for the force): affine, $\mathbf{F} = \mathbf{C}_F \mathbf{x} + \mathbf{D}_F \mathbf{u} + \mathbf{e}_F$, and likewise $\mathbf{F}_n = \mathbf{C}_N \mathbf{x} + \mathbf{D}_N \mathbf{u} + \mathbf{e}_N$ (the $\mathbf{C}$ matrices are zero on the pose).
$$F_i = \frac{K_t\, i_i - c_i F_{n,i}\, r - J_w\, \dot{\omega}_i}{r}, \qquad \dot{\omega}_i = \frac{\left(\mathbf{H}_{drive} \frac{d\mathbf{v}}{dt}\right)_i}{r}$$

**Grip limit** (Coulomb's static friction, with the friction noise's 3σ taken off $\mu_s$), at the start and the end of every step $j$ of the horizon:
$$-\mu_{lim} F_{n,i} \le F_i \le \mu_{lim} F_{n,i}, \qquad \mu_{lim} = \mu_s - 3\sigma_\mu$$
Written in $\mathbf{x}_j$ and $\mathbf{u}_j$, these are a mixed state-input constraint, two half-spaces per wheel, so a convex set:
$$(\mathbf{C}_F - \mu_{lim} \mathbf{C}_N)\, \mathbf{x}_j + (\mathbf{D}_F - \mu_{lim} \mathbf{D}_N)\, \mathbf{u}_j \le \mu_{lim} \mathbf{e}_N - \mathbf{e}_F, \qquad (-\mathbf{C}_F - \mu_{lim} \mathbf{C}_N)\, \mathbf{x}_j + (-\mathbf{D}_F - \mu_{lim} \mathbf{D}_N)\, \mathbf{u}_j \le \mu_{lim} \mathbf{e}_N + \mathbf{e}_F$$
The matrices are worked out before each solve from the estimated state, so within one QP they are constants. It is a hard constraint, not a cost.

**Discretization:** $\frac{d\mathbf{v}}{dt} = \mathbf{A}\mathbf{v} + \mathbf{B}\mathbf{u} + \mathbf{c}$ is held over $T_s$ exactly (zero-order hold, one matrix exponential), together with the distance moved along the robot's axes, $\mathbf{s}$, $\frac{d\mathbf{s}}{dt} = \mathbf{v}$. The pose then moves by $\mathbf{s}$ turned into the room frame at a planned heading $\bar{\theta}_j$ (the last plan's heading, moved on one step):
$$[X, Y]_{j+1} = [X, Y]_j + \mathbf{R}_2(\bar{\theta}_j)\, [s_x, s_y]_j, \qquad \theta_{j+1} = \theta_j + s_{\theta,j}$$
This geometry is the only part of the model that depends on the pose, and the planned heading is its only approximation.

**QP:** minimize $\sum_{j=1}^{N} w_j \|\mathbf{p}_j - \mathbf{p}_j^{ref}\|^2_{\mathbf{Q}_p} + \|\mathbf{v}_j - \mathbf{v}_j^{ref}\|^2_{\mathbf{Q}_v} + \sum_{j=0}^{N-1} r_{\Delta u}\|\mathbf{u}_j - \mathbf{u}_{j-1}\|^2$ (reference velocity turned onto the robot's axes at the reference heading), subject to the model, $|u_{i,j}| \le V_{max}$ and the grip limit. A convex quadratic cost with affine constraints: a convex QP. Solved with OSQP; only $\mathbf{u}_0$ is applied. If a solve fails, the last voltages are held and the failure is counted.

**Velocity estimate:** the pose is measured; the velocity is the model's prediction corrected by the pose prediction error, turned onto the robot's axes: $\hat{\mathbf{v}}_k = \mathbf{v}_{k|k-1} + \kappa\, \mathbf{R}(\theta_k)^\top (\mathbf{p}_k - \mathbf{p}_{k|k-1}) / T_s$.

</details>

In [ ]:
"""
The MPC's model: no slip, with a grip limit
===========================================

AffineGripModel is the robot as this MPC sees it: the wheels roll without slipping, and the motor current
settles at once. Three values that change slowly are held over the horizon (the rolling drag, the roller
force and the turning rate). With them held, everything in the model is affine in the state and input.

Calculates, from the velocity v (along the robot's own axes) and the voltages u:
    - the chassis acceleration:              dv/dt = A v + B u + c
    - the friction force each wheel needs:   F     = C_F v + D_F u + e_F
    - the normal force on each wheel:        F_n   = C_N v + D_N u + e_N

The controller calls:
    held_values(v)                       the values held over the horizon, at the estimated velocity
    affine_model(c_roll, phi_a, turn)    the matrices above
    discretize(model, Ts)                the model held over one controller step Ts
"""


class AffineGripModel:
    """
    The robot with no wheel slip, for the MPC. Built from simulation_parameters, the same values as the plant.
    """

    def __init__(self, params: dict):
        """
        params: simulation_parameters
        """
        self.p = params

        # -------------------------------------------------------------------
        # Fixed geometry and loads, worked out by the plant's robot class from the same parameters.
        # Only fixed values are read from it, never a state.
        # -------------------------------------------------------------------
        geometry = OmniRobotSimulation(params)

        # Drive speeds of the wheels = H_drive v, axle (roller) speeds = H_axial v
        self.H = geometry.H_drive
        self.H_a = geometry.H_axial

        # Normal force on each wheel at rest [N]
        self.F_n_static = np.array([geometry.F_n_front_static, geometry.F_n_front_static,
                                    geometry.F_n_rear_static, geometry.F_n_rear_static])

        # Load transfer (Section 4) as a matrix: F_n = F_n_static + T_load a, with a = [right, forward, yaw]
        pitch = geometry.pitch_gain
        roll_front = geometry.roll_gain_front
        roll_rear = geometry.roll_gain_rear
        self.T_load = np.array([
            [-roll_front, -pitch, 0.0],     # 1 FR
            [roll_front, -pitch, 0.0],      # 2 FL
            [roll_rear, pitch, 0.0],        # 3 RL
            [-roll_rear, pitch, 0.0],       # 4 RR
        ])

        # With no slip the wheels turn with the chassis, so their inertia adds to the chassis's:
        #   M_wheels = M + (Jw / r^2) H_drive^T H_drive
        self.k_wheel = params['Jw'] / params['r']**2
        self.M_wheels = geometry.M + self.k_wheel * self.H.T @ self.H

        # Turning of the robot's own axes: dv/dt = a + theta_dot S v
        self.S = np.array([[0.0, 1.0, 0.0],
                           [-1.0, 0.0, 0.0],
                           [0.0, 0.0, 0.0]])

    # =======================================================================
    # Values held over the horizon
    # =======================================================================

    def held_values(self, v: np.ndarray) -> tuple:
        """
        The three values held over the horizon, worked out at the velocity v.

        v: velocity along the robot's axes [x_dot, y_dot, theta_dot] [m/s, m/s, rad/s]

        Returns (c_roll, phi_a, turn):
            c_roll: rolling drag per unit normal force, Crr tanh(omega / roll_smoothing) (Step 5.3)
            phi_a: roller force per unit normal force, -mu_a (2/pi) atan(k_a v_slip_a) (Step 5.8)
            turn: turning rate theta_dot [rad/s]
        """
        p = self.p

        # Wheel speeds with no slip: omega = H_drive v / r
        omega = self.H @ v / p['r']

        # Rolling drag per unit normal force
        c_roll = p['Crr'] * np.tanh(omega / p['roll_smoothing'])

        # Roller force per unit normal force, from the axle slip v_slip_a = H_axial v
        if p['roller_friction']:
            v_slip_a = self.H_a @ v
            mu_a = p['mu_a_k'] + (p['mu_a_s'] - p['mu_a_k']) * np.exp(-p['alpha_a'] * np.abs(v_slip_a))
            phi_a = -mu_a * (2.0 / np.pi) * np.arctan(p['roller_smoothing'] * v_slip_a)
        else:
            phi_a = np.zeros(4)

        turn = v[2]
        return c_roll, phi_a, turn

    # =======================================================================
    # The model as matrices
    # =======================================================================

    def affine_model(self, c_roll: np.ndarray, phi_a: np.ndarray, turn: float) -> dict:
        """
        The model with the held values, as matrices. Every quantity is written as
        (matrix) v + (matrix) u + (constant), so it stays affine all the way through.

        c_roll, phi_a, turn: from held_values

        Returns a dictionary:
            A, B, c:         dv/dt = A v + B u + c
            C_F, D_F, e_F:   F     = C_F v + D_F u + e_F     friction force each wheel needs [N]
            C_N, D_N, e_N:   F_n   = C_N v + D_N u + e_N     normal force on each wheel [N]
        """
        p = self.p
        r = p['r']
        R = p['R']
        Kt = p['Kt']
        Kb = p['Kb']
        Jw = p['Jw']
        H = self.H
        H_a = self.H_a
        T = self.T_load

        # Step 1: motor current, settled, with the no-slip wheel speed omega = H_drive v / r
        #   I = (u - Kb omega) / R = I_v v + I_u u
        I_v = -Kb * H / (r * R)
        I_u = np.eye(4) / R

        # Step 2: chassis acceleration a (along the robot's axes), with the normal forces of this step.
        # Newton's law with the wheels' inertia added, and the rolling drag and roller force per unit normal force:
        #   M_wheels a = H^T (Kt/r I - c_roll F_n) + H_a^T (phi_a F_n) - k_wheel H^T H (turn S v)
        # with F_n = F_n_static + T_load a. Moving every a to the left gives one matrix L, constant once the
        # values are held:
        #   L a = rhs,   L = M_wheels + H^T diag(c_roll) T_load - H_a^T diag(phi_a) T_load
        L = self.M_wheels + H.T @ np.diag(c_roll) @ T - H_a.T @ np.diag(phi_a) @ T

        # rhs = rhs_v v + rhs_u u + rhs_0
        rhs_v = H.T @ (Kt / r * I_v) - self.k_wheel * turn * H.T @ H @ self.S
        rhs_u = H.T @ (Kt / r * I_u)
        rhs_0 = -H.T @ (c_roll * self.F_n_static) + H_a.T @ (phi_a * self.F_n_static)

        # a = a_v v + a_u u + a_0
        a_v = np.linalg.solve(L, rhs_v)
        a_u = np.linalg.solve(L, rhs_u)
        a_0 = np.linalg.solve(L, rhs_0)

        # Step 3: velocity along the robot's (turning) axes: dv/dt = a + turn S v
        A = a_v + turn * self.S
        B = a_u
        c = a_0

        # Step 4: normal forces, F_n = F_n_static + T_load a
        C_N = T @ a_v
        D_N = T @ a_u
        e_N = self.F_n_static + T @ a_0

        # Step 5: friction force each wheel needs, from the wheel equation (Step 5.7) solved for the force:
        #   F = (Kt I - c_roll F_n r - Jw omega_dot) / r,   omega_dot = H_drive (dv/dt) / r
        C_F = Kt / r * I_v - np.diag(c_roll) @ C_N - Jw / r**2 * H @ A
        D_F = Kt / r * I_u - np.diag(c_roll) @ D_N - Jw / r**2 * H @ B
        e_F = -c_roll * e_N - Jw / r**2 * H @ c

        return {'A': A, 'B': B, 'c': c,
                'C_F': C_F, 'D_F': D_F, 'e_F': e_F,
                'C_N': C_N, 'D_N': D_N, 'e_N': e_N}

    # =======================================================================
    # One controller step
    # =======================================================================

    @staticmethod
    def discretize(model: dict, Ts: float) -> tuple:
        """
        The model held over one controller step Ts, with the voltages held (zero-order hold). Exact for this model.
        Also gives the distance moved along the robot's axes over the step, s, which moves the pose.

        model: from affine_model
        Ts: controller step [s]

        Returns (A_d, B_d, d, S_v, S_u, s_0):
            v_next = A_d v + B_d u + d
            s      = S_v v + S_u u + s_0
        """
        A = model['A']
        B = model['B']
        c = model['c']

        # The velocity v, the distance moved s (ds/dt = v), the held voltages u and a constant 1 go into one
        # matrix, so that one matrix exponential holds the whole step:
        #
        #   d/dt [v, s, u, 1] = [[A, 0, B, c],     v
        #                        [I, 0, 0, 0],     s
        #                        [0, 0, 0, 0],     u (held)
        #                        [0, 0, 0, 0]]     1
        block = np.zeros((11, 11))
        block[0:3, 0:3] = A
        block[0:3, 6:10] = B
        block[0:3, 10] = c
        block[3:6, 0:3] = np.eye(3)
        E = expm(block * Ts)

        A_d = E[0:3, 0:3]
        B_d = E[0:3, 6:10]
        d = E[0:3, 10]
        S_v = E[3:6, 0:3]
        S_u = E[3:6, 6:10]
        s_0 = E[3:6, 10]
        return A_d, B_d, d, S_v, S_u, s_0

### Check: the grip limit is affine

By the definition: a function $f$ is affine when $f(a\,\mathbf{z}_1 + (1-a)\,\mathbf{z}_2) = a f(\mathbf{z}_1) + (1-a) f(\mathbf{z}_2)$ for every $a$, and a set of affine inequalities $f(\mathbf{z}) \le 0$ is convex. The check below takes random velocities, voltages and held values, with $a$ between $-1$ and $2$, and compares both sides for the floor forces and the loads. It also checks that, at the velocity the values are held at, the affine model gives the same forces as the full no-slip equations.

In [ ]:
# ===========================================================================
# Check 1: the friction force and the normal force are affine in v and u
# ===========================================================================
# By the definition, f is affine when f(a z1 + (1 - a) z2) = a f(z1) + (1 - a) f(z2) for any a.
# Here z = (v, u), and f gives [F, F_n] for one set of held values.
check_model = AffineGripModel(simulation_parameters)
rng = np.random.default_rng(0)


def forces(matrices, v, u):
    """[F, F_n] at the velocity v and voltages u, from the matrices of affine_model."""
    F = matrices['C_F'] @ v + matrices['D_F'] @ u + matrices['e_F']
    F_n = matrices['C_N'] @ v + matrices['D_N'] @ u + matrices['e_N']
    return np.concatenate((F, F_n))


largest = 0.0
for case in range(300):
    # One random set of held values, and its matrices
    matrices = check_model.affine_model(*check_model.held_values(rng.normal(0.0, 0.8, 3)))

    # Two random points, and a mixing weight a (also outside 0 to 1)
    v1, v2 = rng.normal(0.0, 1.0, 3), rng.normal(0.0, 1.0, 3)
    u1, u2 = rng.uniform(-12, 12, 4), rng.uniform(-12, 12, 4)
    a = rng.uniform(-1.0, 2.0)

    forces_of_mix = forces(matrices, a * v1 + (1 - a) * v2, a * u1 + (1 - a) * u2)
    mix_of_forces = a * forces(matrices, v1, u1) + (1 - a) * forces(matrices, v2, u2)
    largest = max(largest, np.max(np.abs(forces_of_mix - mix_of_forces)))

print(f"1. Affine by the definition, 300 random cases: largest difference {largest:.1e} N (rounding only)")


# ===========================================================================
# Check 2: the matrices give the same forces as the no-slip equations solved directly
# ===========================================================================
# At the velocity the values are held at, nothing is approximated, so the two must agree.

def forces_from_equations(model, v, u):
    """F and F_n from the no-slip equations, solved directly at (v, u), without the matrices."""
    p = model.p
    c_roll, phi_a, turn = model.held_values(v)

    # Wheel speeds with no slip, and the settled motor current
    omega = model.H @ v / p['r']
    I = (u - p['Kb'] * omega) / p['R']

    # Acceleration with the normal forces of this step (the linear solve of Step 2)
    L = model.M_wheels + model.H.T @ np.diag(c_roll) @ model.T_load - model.H_a.T @ np.diag(phi_a) @ model.T_load
    rhs = (model.H.T @ (p['Kt'] / p['r'] * I - c_roll * model.F_n_static)
           + model.H_a.T @ (phi_a * model.F_n_static)
           - model.k_wheel * model.H.T @ model.H @ (turn * model.S @ v))
    a = np.linalg.solve(L, rhs)

    # Normal forces, wheel acceleration and the friction force each wheel needs
    F_n = model.F_n_static + model.T_load @ a
    omega_dot = model.H @ (a + turn * model.S @ v) / p['r']
    F = (p['Kt'] * I - c_roll * F_n * p['r'] - p['Jw'] * omega_dot) / p['r']
    return F, F_n


largest = 0.0
for case in range(200):
    v = rng.normal(0.0, 0.8, 3)
    u = rng.uniform(-10, 10, 4)
    matrices = check_model.affine_model(*check_model.held_values(v))
    F, F_n = forces_from_equations(check_model, v, u)
    largest = max(largest, np.max(np.abs(forces(matrices, v, u) - np.concatenate((F, F_n)))))

print(f"2. Matrices against the equations at the held velocity: largest difference {largest:.1e} N")